# Initialization

In [1]:
import pandas as pd
import ast
import os
from tqdm import tqdm

tqdm.pandas()

# Load Impact DF

In [2]:
def try_literal_eval(x):
    if isinstance(x, str):
        x = x.strip()
        if (x.startswith("[") and x.endswith("]")) or \
           (x.startswith("{") and x.endswith("}")) or \
           (x.startswith("(") and x.endswith(")")):
            try:
                return ast.literal_eval(x)
            except (ValueError, SyntaxError):
                return x
    return x

In [3]:
dataset_dir = "/home/yishin/keith/patent_research/model_io"

In [8]:
df = pd.read_csv(os.path.join(dataset_dir, "1_Impact_Sub.csv"), encoding="utf-8")
df = df.map(try_literal_eval)

df.head()

,title,caption,image_paths,best_fig_desc,Loc_class,main_class,sub_class
0,Cupcake with contrasting icing,The image is a black and white picture of a cu...,[impact_dataset/2022/USD0957089-20220712/USD09...,{'FIG. 1': 'FIG. 1 is a perspective view of a ...,"{11-01, 01-01}",1,1
1,Ring with an integrated spoon,The image is a drawing of a ring with an integ...,[impact_dataset/2022/USD0966130-20221011/USD09...,{'FIG. 1': 'FIG. 1 is a front perspective view...,"{11-01, 01-01}",1,1
2,Butter stick,The image is a white and black drawing of a bu...,[impact_dataset/2022/USD0962586-20220906/USD09...,"{'FIG. 1': 'FIG. 1 is a front, top and left si...","{01-06, 11-01}",1,6
3,Necklace,"The image is a round shape, and the necklace i...",[impact_dataset/2022/USD0962109-20220830/USD09...,{'FIG. 3': 'FIG. 3 is a right side perspective...,"{11-01, 01-01}",1,1
4,Dietary supplement,"The image is circular in shape, and it represe...",[impact_dataset/2022/USD0941457-20220118/USD09...,{'FIG. 1': 'FIG. 1 is a top perspective view o...,"{11-01, 01-01}",1,1


# Formula Method

## Select Best Figures

In [14]:
import pytesseract
import re
import numpy as np
import cv2
from pathlib import Path
from PIL import Image, ImageOps

In [15]:
def preprocess_for_ocr(image: Image.Image, min_height: int, dilate: bool = True) -> Image.Image:
    image = image.convert("L")
    image = ImageOps.autocontrast(image)

    w, h = image.size
    scale = max(1.0, min_height / h)
    if scale > 1.0:
        image = image.resize((int(w * scale), int(h * scale)), Image.Resampling.LANCZOS)

    arr = np.array(image)

    # Denoise before binarization — removes scanner noise without blurring text
    arr = cv2.fastNlMeansDenoising(arr, h=15)

    # Otsu picks the threshold automatically from the image histogram
    _, arr = cv2.threshold(arr, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    if dilate:
        kernel = np.ones((2, 2), np.uint8)
        arr = cv2.dilate(arr, kernel, iterations=1)

    # Padding helps Tesseract with text near the image edge
    arr = cv2.copyMakeBorder(arr, 20, 20, 20, 20, cv2.BORDER_CONSTANT, value=255)

    return Image.fromarray(arr)

In [16]:
_FIG_LABEL = re.compile(
    r'\bF[I1lL|]G[S]?[.\s_\-·,]?\s*(\d{1,3}\s*[A-Za-z]?)\b',
    re.IGNORECASE,
)
_CONFIGS = [r'--oem 3 --psm 11', r'--oem 3 --psm 6', r'--oem 3 --psm 3']

In [17]:
def extract_fig_labels(
    img_path: str,
    min_height: int = 800,
) -> tuple[list[tuple[str, int]], int]:
    """
    Returns (matched_labels, total_fig_count).

    matched_labels : list[(fig_key, rotation)]
        All unique FIG labels detected, paired with the rotation that found them.

    total_fig_count : int
        Total number of distinct FIG labels found in the image on the
        successful pass. Useful for knowing how many figures share a page.
    """
    pil_img = Image.open(img_path)

    for rotation in [0, 90, 270]:
        rotated   = pil_img.rotate(rotation, expand=True) if rotation else pil_img
        processed = preprocess_for_ocr(rotated, min_height=min_height)

        for cfg in _CONFIGS:
            raw_text     = pytesseract.image_to_string(processed, config=cfg)
            normalized   = " ".join(raw_text.split())

            # Collect every unique FIG label found in this pass.
            all_keys_seen: dict[str, tuple[str, int]] = {}
            for m in _FIG_LABEL.finditer(normalized):
                num = re.sub(r'\s+', '', m.group(1)).upper()
                key = f"FIG. {num}"
                if key not in all_keys_seen:
                    all_keys_seen[key] = (key, rotation)

            if all_keys_seen:
                found           = list(all_keys_seen.values())
                total_fig_count = len(all_keys_seen)
                return found, total_fig_count

    return [], 0

In [18]:
def match_figs_by_ocr(
    image_paths,
    best_fig_desc,
    verbose: bool = False,
) -> dict:
    """
    Returns dict mapping FIG key -> {'path': ..., 'rotation': ..., 'fig_count': ...}

    fig_count is the total number of FIG labels detected on that image page,
    which indicates how many figures share the same sheet.
    """
    if isinstance(image_paths, str):
        image_paths = ast.literal_eval(image_paths)
    if isinstance(best_fig_desc, str):
        best_fig_desc = ast.literal_eval(best_fig_desc)

    target_keys = set(best_fig_desc.keys())
    result      = {}

    for path in image_paths:
        if not Path(path).exists():
            if verbose:
                print(f"  [SKIP] missing file: {path}")
            continue

        try:
            found_labels, total_fig_count = extract_fig_labels(path)
        except Exception as e:
            if verbose:
                print(f"  [ERROR] {path}: {e}")
            continue

        for (label, rotation) in found_labels:
            if label in target_keys and label not in result:
                result[label] = {
                    "path":      path,
                    "rotation":  rotation,
                    "fig_count": total_fig_count,
                }
                if verbose:
                    print(
                        f"  [MATCH] {label} → {Path(path).name} "
                        f"(rotation={rotation}°, page_figs={total_fig_count})"
                    )

        if result.keys() == target_keys:
            break

    return result

In [19]:
def random_row_index(df):
    return np.random.randint(0, len(df))

idx = random_row_index(Impact_df)

In [20]:
idx

928

In [21]:
Impact_df['title'].iloc[idx]

'Soil sensor module for a soil sensing system'

In [22]:
Impact_df['best_fig_desc'].iloc[idx]

{'FIG. 1': 'FIG. 1 is a front and bottom perspective view of a soil sensor module for a soil sensing system.',
 'FIG. 2': 'FIG. 2 is a front view thereof.',
 'FIG. 4': 'FIG. 4 is a left side view thereof.',
 'FIG. 5': 'FIG. 5 is a right side view thereof; and,'}

In [23]:
best_figs = match_figs_by_ocr(Impact_df['image_paths'].iloc[idx], Impact_df['best_fig_desc'].iloc[idx], verbose=True)
print(best_figs)

{}


## Figure Isolation

In [24]:
import re
import cv2
import numpy as np
import pytesseract
from pathlib import Path
import math
from PIL import Image
from collections import defaultdict
from typing import Dict, List, Optional, Tuple, Union

In [25]:
def _detect_fig_num_in_crop(crop_bgr: np.ndarray, min_height: int = 800) -> str | None:
    """
    OCR a single BGR crop and return a normalised key like 'FIG. 3', or None.
    Tries PSM 11 (sparse) first, then 6 and 3 as fallbacks.
    """
    pil = Image.fromarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
    processed = preprocess_for_ocr(pil, min_height=min_height)

    for cfg in ["--oem 3 --psm 11", "--oem 3 --psm 6", "--oem 3 --psm 3"]:
        text = pytesseract.image_to_string(processed, config=cfg)
        m    = _FIG_LABEL.search(" ".join(text.split()))
        if m:
            num = re.sub(r'\s+', '', m.group(1)).upper()
            return f"FIG. {num}"
    return None

In [26]:
def merge_figure_boxes(
    boxes: List[Tuple[int, int, int, int]],
    box_count: int,
    mode: str = "edge",
) -> List[Tuple[int, int, int, int]]:
    """
    Greedily merge bounding boxes until exactly box_count remain.
    """
    if box_count <= 0:
        raise ValueError("box_count must be a positive integer.")
    if len(boxes) <= box_count:
        return list(boxes)

    def _edge_dist(a: tuple, b: tuple) -> float:
        dx = max(0, max(a[0], b[0]) - min(a[0] + a[2], b[0] + b[2]))
        dy = max(0, max(a[1], b[1]) - min(a[1] + a[3], b[1] + b[3]))
        return math.hypot(dx, dy)

    def _center_dist(a: tuple, b: tuple) -> float:
        return math.hypot(
            (a[0] + a[2] / 2) - (b[0] + b[2] / 2),
            (a[1] + a[3] / 2) - (b[1] + b[3] / 2),
        )

    def _union(a: tuple, b: tuple) -> Tuple[int, int, int, int]:
        x = min(a[0], b[0])
        y = min(a[1], b[1])
        x2 = max(a[0] + a[2], b[0] + b[2])
        y2 = max(a[1] + a[3], b[1] + b[3])
        return (x, y, x2 - x, y2 - y)

    dist_fn = _edge_dist if mode == "edge" else _center_dist
    pool = list(boxes)

    while len(pool) > box_count:
        best_i, best_j, best_d = 0, 1, float("inf")

        for i in range(len(pool)):
            for j in range(i + 1, len(pool)):
                d = dist_fn(pool[i], pool[j])
                if d < best_d:
                    best_d, best_i, best_j = d, i, j

        merged = _union(pool[best_i], pool[best_j])
        pool = [b for k, b in enumerate(pool) if k not in (best_i, best_j)]
        pool.append(merged)

    pool.sort(key=lambda b: (b[1], b[0]))
    return pool

In [27]:
def _sanitize(name: str) -> str:
    """Strip filesystem-unsafe characters from a folder/file name."""
    return re.sub(r'[<>:"/\\|?*\x00-\x1f]', "_", name).strip()

In [28]:
def _save_crop(crop_bgr: np.ndarray, out_dir: Path, fig_label: str) -> Path:
    """Save a BGR crop as JPEG and return its path."""
    filename = _sanitize(fig_label.replace(". ", "_").replace(" ", "_")) + ".jpg"
    out_path = out_dir / filename
    cv2.imwrite(str(out_path), crop_bgr)
    return out_path

In [29]:
def detect_and_merge_boxes(
    img_bgr:   np.ndarray,
    fig_count: int,
    min_area:  int = 500,
    padding:   int = 25,
    merge_gap: int = 35,
    invert:    bool = True,
) -> list[tuple[int, int, int, int]]:
    """
    Find content bounding boxes via whitespace separation, then merge down
    to exactly fig_count boxes using merge_figure_boxes().
    """
    gray    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (3, 3), 0)
 
    thresh_type = cv2.THRESH_BINARY_INV if invert else cv2.THRESH_BINARY
    binary  = cv2.threshold(blurred, 0, 255, thresh_type + cv2.THRESH_OTSU)[1]
    kernel  = cv2.getStructuringElement(cv2.MORPH_RECT, (merge_gap, merge_gap))
    dilated = cv2.dilate(binary, kernel, iterations=1)
 
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
 
    h_img, w_img = img_bgr.shape[:2]
    boxes = []
    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        if w * h < min_area:
            continue
        x1 = max(x - padding, 0);  y1 = max(y - padding, 0)
        x2 = min(x + w + padding, w_img);  y2 = min(y + h + padding, h_img)
        boxes.append((x1, y1, x2 - x1, y2 - y1))
 
    boxes.sort(key=lambda b: (b[1], b[0]))
 
    if len(boxes) > fig_count:
        boxes = merge_figure_boxes(boxes, box_count=fig_count)
        boxes.sort(key=lambda b: (b[1], b[0]))
 
    return boxes

In [30]:
def rotate_cv2(img: np.ndarray, angle: int) -> np.ndarray:
    """Rotate an OpenCV array counter-clockwise by angle degrees."""
    if angle == 0:
        return img
    pil = Image.fromarray(img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    pil = pil.rotate(angle, expand=True)
    arr = np.array(pil)
    return arr if img.ndim == 2 else cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)

In [31]:
def assign_crops_to_figs(
    fig_dict:   dict,
    title:      str,
    output_root: str  = "figure_crops",
    min_area:   int   = 500,
    padding:    int   = 25,
    merge_gap:  int   = 35,
    invert:     bool  = True,
    min_height: int   = 800,
    verbose:    bool  = False,
) -> dict:
    """
    For each FIG entry in fig_dict, detect its bounding box, extract the crop,
    save it to ``{output_root}/{title}/``, and attach the path to the result.
 
    Parameters
    ----------
    fig_dict : dict
        Output of match_figs_by_ocr — keys are FIG labels, values contain
        'path', 'rotation', and 'fig_count'.
 
    title : str
        Used as the subfolder name under output_root.  Filesystem-unsafe
        characters are replaced with underscores.
 
    output_root : str
        Root directory for saved crops.  Defaults to ``"figure_crops"``.
 
    Returns
    -------
    dict
        Same structure as fig_dict with four extra keys per entry:
        'box'        — (x, y, w, h) in the rotated image's coordinate space,
        'crop'       — BGR numpy array,
        'saved_path' — Path where the crop was written, or None if unmatched.
    """
    # Build output directory: figure_crops/{title}/
    out_dir = Path(output_root) / _sanitize(title)
    out_dir.mkdir(parents=True, exist_ok=True)
 
    if verbose:
        print(f"Saving crops to: {out_dir}")
 
    result = {
        fig: {**meta, "box": None, "crop": None, "saved_path": None}
        for fig, meta in fig_dict.items()
    }
 
    # Group FIG labels by the image page they share.
    page_groups: dict[tuple, list[str]] = defaultdict(list)
    for fig_label, meta in fig_dict.items():
        page_key = (meta["path"], meta["rotation"], meta["fig_count"])
        page_groups[page_key].append(fig_label)
 
    for (path, rotation, fig_count), fig_labels in page_groups.items():
 
        if not Path(path).exists():
            if verbose:
                print(f"  [SKIP] missing: {path}")
            continue
 
        raw = cv2.imread(path)
        if raw is None:
            if verbose:
                print(f"  [ERROR] could not read: {path}")
            continue
 
        img = rotate_cv2(raw, rotation)
 
        # --- Fast path: single figure on this page ---
        if fig_count == 1:
            h_img, w_img = img.shape[:2]
            box       = (0, 0, w_img, h_img)
            crop      = img
            fig_label = fig_labels[0]
 
            saved = _save_crop(crop, out_dir, fig_label)
 
            result[fig_label]["box"]        = box
            result[fig_label]["crop"]       = crop
            result[fig_label]["saved_path"] = saved
 
            if verbose:
                print(f"\n  {Path(path).name}  rotation={rotation}°  "
                      f"fig_count=1 → full image assigned to {fig_label}  "
                      f"→ saved {saved.name}")
            continue
 
        # --- Multi-figure page: detect, merge, identify ---
        boxes = detect_and_merge_boxes(img, fig_count, min_area, padding, merge_gap, invert)
        crops = [img[y: y + h, x: x + w] for x, y, w, h in boxes]
 
        if verbose:
            print(f"\n  {Path(path).name}  rotation={rotation}°  "
                  f"target_figs={fig_count}  boxes_found={len(boxes)}")
 
        # OCR pass: read the FIG label inside each crop.
        ocr_map: dict[str, int] = {}
        for i, crop in enumerate(crops):
            label = _detect_fig_num_in_crop(crop, min_height=min_height)
            if label and label not in ocr_map:
                ocr_map[label] = i
                if verbose:
                    print(f"    crop[{i}] → OCR detected {label}")
 
        # Positional fallback for any labels OCR could not identify.
        def _fig_sort_key(lbl: str) -> int:
            m = re.search(r'(\d+)', lbl)
            return int(m.group(1)) if m else 0
 
        unmatched_labels = [l for l in sorted(fig_labels, key=_fig_sort_key)
                            if l not in ocr_map]
        unused_indices   = [i for i in range(len(crops))
                            if i not in ocr_map.values()]
 
        for label, idx in zip(unmatched_labels, unused_indices):
            ocr_map[label] = idx
            if verbose:
                print(f"    crop[{idx}] → positional fallback for {label}")
 
        # Write results and save crops.
        for fig_label in fig_labels:
            if fig_label not in ocr_map:
                if verbose:
                    print(f"    [UNMATCHED] {fig_label}")
                continue
 
            idx      = ocr_map[fig_label]
            x, y, w, h = boxes[idx]
            crop     = crops[idx]
            saved    = _save_crop(crop, out_dir, fig_label)
 
            result[fig_label]["box"]        = (x, y, w, h)
            result[fig_label]["crop"]       = crop
            result[fig_label]["saved_path"] = saved
 
            if verbose:
                print(f"    {fig_label} → saved {saved.name}")
 
    return result

In [32]:
best_figs

{}

In [33]:
result = assign_crops_to_figs(best_figs, title="temp", verbose=True)

Saving crops to: figure_crops/temp
